# Solutions – Python Decorators 

---
## Exercise 1.1 – Simple Logging Decorator

**Goal:** Write a decorator `log_call` that prints the function name every time it is called.

Requirements:
- Before the wrapped function runs, print: `Calling <function_name>`
- After it finishes, print: `<function_name> finished`

Example:
```python
@log_call
def greet(name):
    print(f"Hello {name}")
```


In [ ]:
def log_call(func):
    """Simple logging decorator that reports before and after a call.

    This version does not yet preserve metadata (that's Exercise 1.2).
    """
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}")
        result = func(*args, **kwargs)
        print(f"{func.__name__} finished")
        return result
    return wrapper

@log_call
def greet(name):
    print(f"Hello {name}")

# Test
greet("Alice")


### Exercise 1.2 – Preserve Function Metadata

Now we extend `log_call` using `functools.wraps` so the wrapped function keeps `__name__` and `__doc__` of the original function.

Key idea:

```python
import functools

def decorator(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        ...
    return wrapper
```


In [ ]:
import functools

def log_call(func):
    """Logging decorator that preserves metadata using functools.wraps."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}")
        result = func(*args, **kwargs)
        print(f"{func.__name__} finished")
        return result
    return wrapper

@log_call
def greet_with_doc(name):
    """Greet a user by name."""
    print(f"Hello {name}")

# Test
greet_with_doc("Bob")
print("__name__:", greet_with_doc.__name__)
print("__doc__:", greet_with_doc.__doc__)


---
## Exercise 2.1 – Repeat Decorator

Write a decorator factory `repeat(n)` that runs the decorated function `n` times.

Requirements:
- `repeat` takes `n` and returns a decorator.
- The wrapper must accept `*args, **kwargs` and pass them along.

Example:
```python
@repeat(3)
def say_hi():
    print("Hi")
```

In [ ]:
def repeat(n):
    """Decorator factory that repeats a function call n times."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(n):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(3)
def say_hi():
    print("Hi")

# Test
say_hi()


### Exercise 2.2 – Timed Repeats

Extend `repeat(n)` so that it also measures the total time and prints it after all repetitions.

Example output:

```text
Hello
Hello
Hello
Completed say_hello 3 times in 0.0012s
```

We can use `time.perf_counter()` for higher precision.


In [ ]:
import time

def repeat(n):
    """Decorator factory that repeats a function n times and times the total duration."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            start = time.perf_counter()
            result = None
            for _ in range(n):
                result = func(*args, **kwargs)
            end = time.perf_counter()
            duration = end - start
            print(f"Completed {func.__name__} {n} times in {duration:.4f}s")
            return result
        return wrapper
    return decorator

@repeat(3)
def say_hello():
    print("Hello")

# Test
say_hello()


---
## Exercise 3.1 – Type Check for Integer Arguments (`only_ints`)

Write a decorator `only_ints` that ensures all **positional arguments** are `int`.

Behavior:
- If any positional arg is not an `int`, raise `TypeError`.
- Otherwise call the function normally.

Example:
```python
@only_ints
def add(a, b):
    return a + b
```

In [ ]:
def only_ints(func):
    """Decorator that allows only integer positional arguments.

    If any positional argument is not an int, raise TypeError.
    """
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        for arg in args:
            if not isinstance(arg, int):
                raise TypeError(f"Argument {arg!r} is not int (got {type(arg).__name__})")
        # Optionally, also check kwargs.values() if desired
        return func(*args, **kwargs)
    return wrapper

@only_ints
def add(a, b):
    return a + b

# Tests
print(add(1, 2))

try:
    add(1, "x")
except TypeError as e:
    print("Caught TypeError as expected:", e)


## Exercise 3.2 – Ensure Non-Negative Result (`non_negative`)

Write a decorator `non_negative` that:

- Calls the function.
- If the result is **negative**, returns `0` instead.
- Otherwise returns the original result.

Example:
```python
@non_negative
def subtract(a, b):
    return a - b
```

In [ ]:
def non_negative(func):
    """Decorator that clamps a numeric result at zero (no negatives).

    If the function's result is < 0, return 0 instead.
    """
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        if isinstance(result, (int, float)) and result < 0:
            return 0
        return result
    return wrapper

@non_negative
def subtract(a, b):
    return a - b

# Tests
print(subtract(10, 3))   # 7
print(subtract(3, 10))   # 0 instead of -7


---
# Summary

In these solutions you saw:

- How to write simple decorators and use `functools.wraps`.
- How to write decorator factories that accept parameters (like `repeat(n)`).
- How to perform argument validation in decorators (`only_ints`).
- How to post-process function results (`non_negative`).

You can now compare these implementations with your own and modify them to explore other behaviors.
